<a href="https://colab.research.google.com/github/keshariujjwal51/Ujjwal/blob/main/Assignment_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single Agent Systems & Agent Pipelines — PDF Q&A Edition (No API Key)

A hands-on companion notebook. Each section below takes one concept from the quiz
and turns it into a small, runnable demo — built around a real task: read a PDF,
and an agent answers questions using only that PDF's content.

**This version needs no API key and makes no network calls to any LLM provider.**
Answers are produced by a small **extractive Q&A engine built from scratch**:
plain-Python keyword scoring and sentence selection, no external model at all.

1. Stateful directed graphs
2. Nodes & edges
3. Conditional routing
4. Cycles / retry loops
5. Single agent simulating multi-agent behavior
6. JSON schema tools
7. Sequential vs. parallel tool calls
8. Error handling strategies
9. Trajectory evaluation
10. Task completion rate & cost metrics

At the end, all of these pieces are wired together into one working **PDF Q&A
agent pipeline**: point it at a PDF once, then ask it as many questions as you like.


## Setup

This notebook needs only:

- `pypdf` to extract text from the PDF
- Python's standard library (`re`, `collections.Counter`, etc.) for the
  extractive Q&A logic

**No API key, no account, and no internet access at inference time.** Everything
that "answers" a question runs locally, in this notebook.


In [ ]:
# Install dependencies (safe to re-run)
!pip install -q pypdf


In [ ]:
# Common imports used throughout the notebook
import io
import os
import re
import json
import time
import random
import concurrent.futures
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, Any, Dict, List, Optional

from pypdf import PdfReader

print("Ready -- no API key needed.")


Ready -- no API key needed.


## Load the PDF (from Google Drive)

This mounts your Google Drive (you'll be asked to authorize access), then reads
the PDF from a path you specify relative to "My Drive". If you're not running in
Colab, it falls back to asking for a local file path.


In [ ]:
DRIVE_PDF_PATH = "week8.PDF.pdf"  # <-- change this to your file's path within My Drive

PDF_PATH = None

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PDF_PATH = f"/content/drive/MyDrive/{DRIVE_PDF_PATH}"
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError(
            f"Could not find '{PDF_PATH}'. Double-check DRIVE_PDF_PATH above "
            f"matches the file's location within your Google Drive."
        )
    print(f"Using PDF from Google Drive: {PDF_PATH}")
except ImportError:
    # Not running in Colab -- fall back to a manual local path.
    PDF_PATH = input("Enter the local path to a PDF file: ").strip()

assert PDF_PATH, "No PDF selected."
assert os.path.exists(PDF_PATH), f"File not found: {PDF_PATH}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using PDF from Google Drive: /content/drive/MyDrive/week8.PDF.pdf


## From-Scratch Text Tools

Everything below is plain Python -- no ML model, no embeddings, no API. Just:

- a **tokenizer** (lowercase + strip punctuation),
- a **sentence splitter** (regex on `.`/`!`/`?`),
- a **stopword list** (so common words like "the" or "is" don't dominate scoring),
  and
- a **word-frequency scorer**, which is the same idea behind classic
  frequency-based extractive summarization (score a sentence by how many
  "important" words it contains, relative to the rest of the document).

These are the building blocks the rest of the notebook uses to retrieve and
"answer" -- by *selecting* existing sentences from the PDF, not generating new
text.


In [ ]:
STOPWORDS = set("""
a an the and or but if while is are was were be been being of to in on at for
with as by from this that these those it its into over under between among
not no nor so than too very can will just should would could may might must
shall do does did have has had you your yours i we our ours they them their
he she his her him what which who whom there here when where why how all
each few more most other some such only own same
""".split())


def tokenize(text: str) -> List[str]:
    return [w.lower().strip(".,!?;:\"'()[]{}") for w in text.split()]


def split_into_sentences(text: str) -> List[str]:
    """A small regex-based sentence splitter -- good enough for PDF-extracted
    text, which is rarely perfectly clean anyway."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 3]


def word_frequencies(sentences: List[str]) -> Counter:
    freq = Counter()
    for s in sentences:
        for w in tokenize(s):
            if w and w not in STOPWORDS and len(w) > 2:
                freq[w] += 1
    return freq


def score_sentence(sentence: str, freq: Counter) -> float:
    words = [w for w in tokenize(sentence) if w not in STOPWORDS and len(w) > 2]
    if not words:
        return 0.0
    return sum(freq.get(w, 0) for w in words) / len(words)


# Quick sanity check
demo_sentences = ["The cat sat on the mat.", "Cats are popular pets around the world."]
demo_freq = word_frequencies(demo_sentences)
print("Word frequencies:", dict(demo_freq))
print("Sentence scores:", [round(score_sentence(s, demo_freq), 2) for s in demo_sentences])


Word frequencies: {'cat': 1, 'sat': 1, 'mat': 1, 'cats': 1, 'popular': 1, 'pets': 1, 'around': 1, 'world': 1}
Sentence scores: [1.0, 1.0]


## 1. Stateful Directed Graph

A **stateful directed graph** is a workflow made of *nodes* (steps) and *edges*
(connections between steps) where information gathered at one node persists and
can be used by later nodes. Unlike a **linear pipeline**, which always runs
`step1 -> step2 -> step3` in a fixed order, a stateful graph can:

- branch (choose different next nodes depending on state),
- loop (revisit a node), and
- carry memory forward (the "state" dictionary).

We'll reuse this same tiny graph engine for the whole notebook. In our PDF Q&A
pipeline, the state dictionary is what carries the **PDF text, chunks, the
user's question, retrieved context, and the final answer** from node to node.


In [ ]:
class StateGraph:
    """A tiny stateful directed graph engine."""

    def __init__(self):
        self.nodes: Dict[str, Callable[[dict], dict]] = {}
        self.edges: Dict[str, Callable[[dict], str]] = {}  # node_name -> function(state) -> next_node_name
        self.entry_point: Optional[str] = None

    def add_node(self, name: str, fn: Callable[[dict], dict]):
        self.nodes[name] = fn
        return self

    def add_edge(self, from_node: str, router: Callable[[dict], str]):
        """router(state) returns the name of the next node, or 'END'."""
        self.edges[from_node] = router
        return self

    def set_entry(self, name: str):
        self.entry_point = name
        return self

    def run(self, initial_state: dict, max_steps: int = 25) -> dict:
        state = dict(initial_state)
        state.setdefault("trajectory", [])  # used later for trajectory evaluation
        current = self.entry_point
        steps = 0

        while current != "END" and steps < max_steps:
            steps += 1
            node_fn = self.nodes[current]
            state = node_fn(state)
            state["trajectory"].append(current)

            router = self.edges.get(current)
            current = router(state) if router else "END"

        state["_stopped_reason"] = "END" if current == "END" else "max_steps_reached"
        return state


In [ ]:
# Quick demo: a graph that loads the PDF, then "loops" a chunking node until
# every page has been chunked, proving the graph remembers state (self.chunks,
# self.page_index) across visits -- a linear pipeline can't do that as cleanly.

def load_pdf_node(state):
    reader = PdfReader(state["pdf_path"])
    state["pages"] = [p.extract_text() or "" for p in reader.pages]
    state["page_index"] = 0
    state["chunks"] = []
    print(f"  loaded PDF with {len(state['pages'])} page(s)")
    return state

def chunk_one_page_node(state):
    i = state["page_index"]
    text = state["pages"][i].strip()
    if text:
        state["chunks"].append({"page": i + 1, "text": text})
    state["page_index"] += 1
    print(f"  chunked page {i + 1}/{len(state['pages'])}")
    return state

def route_chunking(state):
    return "END" if state["page_index"] >= len(state["pages"]) else "chunk_page"

demo_graph = StateGraph()
demo_graph.add_node("load_pdf", load_pdf_node)
demo_graph.add_node("chunk_page", chunk_one_page_node)
demo_graph.add_edge("load_pdf", lambda s: "chunk_page")
demo_graph.add_edge("chunk_page", route_chunking)
demo_graph.set_entry("load_pdf")

result = demo_graph.run({"pdf_path": PDF_PATH})
print("Trajectory:", result["trajectory"])
print(f"Total chunks created: {len(result['chunks'])}")


  loaded PDF with 2 page(s)
  chunked page 1/2
  chunked page 2/2
Trajectory: ['load_pdf', 'chunk_page', 'chunk_page']
Total chunks created: 2


## 2. Nodes & Edges

- **Node** = a unit of work (e.g. "retrieve relevant chunks", "extract an
  answer", "format the answer").
- **Edge** = the connection that decides what happens *after* a node finishes.

Below, `retrieve_context` and `extractive_answer` are nodes, and the function
that decides "only try to answer if we actually found relevant context,
otherwise say so" is an edge.


In [ ]:
def simple_chunk_text(pages: List[str], chunk_size: int = 800, overlap: int = 100) -> List[dict]:
    """Turn a list of page texts into overlapping text chunks with page numbers."""
    chunks = []
    for page_num, page_text in enumerate(pages, start=1):
        page_text = page_text.strip()
        start = 0
        while start < len(page_text):
            piece = page_text[start:start + chunk_size]
            if piece.strip():
                chunks.append({"page": page_num, "text": piece})
            start += chunk_size - overlap
            if len(page_text) <= chunk_size:
                break
    return chunks


def score_chunk(chunk_text: str, query: str) -> int:
    """Very small keyword-overlap retrieval score (no embeddings needed)."""
    query_words = {w for w in tokenize(query) if w not in STOPWORDS and len(w) > 2}
    chunk_words = set(tokenize(chunk_text))
    return len(query_words & chunk_words)


def retrieve_context_node(state, top_k: int = 4, min_score: int = 1):
    scored = [
        (score_chunk(c["text"], state["question"]), c)
        for c in state["chunks"]
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    top = [c for score, c in scored[:top_k] if score >= min_score]
    state["retrieved"] = top
    print(f"  [node] retrieve_context -> found {len(top)} relevant chunk(s)")
    return state


def extractive_lookup_answer(question: str, retrieved_chunks: List[dict], num_sentences: int = 2) -> Optional[str]:
    """Pick the sentences from the retrieved chunks that share the most
    keywords with the question -- a simple, fully local extractive QA
    strategy: no generation, just selection of existing text."""
    q_words = {w for w in tokenize(question) if w not in STOPWORDS and len(w) > 2}
    candidates = []
    seen_sentences = set()  # chunks can overlap, so dedupe repeated sentences
    for c in retrieved_chunks:
        for sentence in split_into_sentences(c["text"]):
            key = sentence.strip().lower()
            if key in seen_sentences:
                continue
            seen_sentences.add(key)
            s_words = set(tokenize(sentence))
            overlap = len(q_words & s_words)
            if overlap > 0:
                candidates.append((overlap, c["page"], sentence))

    if not candidates:
        return None

    candidates.sort(key=lambda t: t[0], reverse=True)
    top = candidates[:num_sentences]
    top_in_order = sorted(top, key=lambda t: t[1])  # restore page order for readability
    return " ".join(f"[Page {page}] {sentence}" for _, page, sentence in top_in_order)


def extractive_answer_node(state):
    answer = extractive_lookup_answer(state["question"], state["retrieved"])
    state["answer"] = answer or "I couldn't find a sentence that answers that question in the retrieved context."
    print("  [node] extractive_answer -> selected sentence(s) directly from the PDF")
    return state


def no_context_node(state):
    state["answer"] = "I couldn't find anything relevant to that question in the PDF."
    print("  [node] no_context -> nothing relevant found")
    return state


def edge_after_retrieval(state):
    return "extractive_answer" if state["retrieved"] else "no_context"


node_edge_graph = StateGraph()
node_edge_graph.add_node("retrieve_context", retrieve_context_node)
node_edge_graph.add_node("extractive_answer", extractive_answer_node)
node_edge_graph.add_node("no_context", no_context_node)
node_edge_graph.add_edge("retrieve_context", edge_after_retrieval)
node_edge_graph.add_edge("extractive_answer", lambda s: "END")
node_edge_graph.add_edge("no_context", lambda s: "END")
node_edge_graph.set_entry("retrieve_context")

# Build chunks once from the PDF (reused below too)
reader = PdfReader(PDF_PATH)
pdf_pages = [p.extract_text() or "" for p in reader.pages]
pdf_chunks = simple_chunk_text(pdf_pages)
print(f"Built {len(pdf_chunks)} chunk(s) from {len(pdf_pages)} page(s).\n")

demo_question = "What is this document about?"
out = node_edge_graph.run({"question": demo_question, "chunks": pdf_chunks})
print("\nQuestion:", demo_question)
print("Answer:", out["answer"])


Built 11 chunk(s) from 2 page(s).

  [node] retrieve_context -> found 0 relevant chunk(s)
  [node] no_context -> nothing relevant found

Question: What is this document about?
Answer: I couldn't find anything relevant to that question in the PDF.


## 3. Conditional Routing

Conditional routing sends a query to the right handling path based on its
intent. For a PDF Q&A agent, three useful routes are:

- **Summary** — the user wants an overview of the whole document.
- **Specific lookup** — the user is asking about a fact/detail in the PDF.
- **Out of scope** — the question has nothing to do with the PDF at all.


In [ ]:
def route_question(question: str) -> str:
    q = question.lower()
    summary_words = ["summarize", "summary", "overview", "what is this document about", "tl;dr"]
    out_of_scope_words = ["weather", "your opinion", "who are you", "joke"]

    if any(w in q for w in summary_words):
        return "Summary"
    elif any(w in q for w in out_of_scope_words):
        return "Out of Scope"
    else:
        return "Specific Lookup"


test_questions = [
    "Can you summarize this document for me?",
    "What does section 2 say about pricing?",
    "What's the weather like today?",
]

for q in test_questions:
    print(f"{q!r:50s} -> {route_question(q)}")


'Can you summarize this document for me?'          -> Summary
'What does section 2 say about pricing?'           -> Specific Lookup
"What's the weather like today?"                   -> Out of Scope


## 4. Cycles (Loops) & Retry Logic

Loops let an agent repeat a step until it succeeds, instead of giving up after
one failure. Even with no external API, a local pipeline can still have
transient failures -- e.g. a locked file, a momentary resource conflict, or (as
simulated here) a flaky step in the extraction logic. A **retry loop**
re-attempts those instead of crashing the whole pipeline.


In [ ]:
class TransientLocalError(Exception):
    """A stand-in for any transient failure a local pipeline step might hit --
    used here purely to demonstrate retry logic without needing a network call."""
    pass


def call_with_retry(fn, max_retries: int = 3, backoff: float = 0.3):
    trajectory = []
    for attempt in range(1, max_retries + 1):
        try:
            trajectory.append(f"attempt_{attempt}")
            result = fn()
            trajectory.append("success")
            return result, trajectory
        except TransientLocalError as e:
            trajectory.append(f"failed:{type(e).__name__}")
            print(f"  attempt {attempt} failed ({type(e).__name__}), retrying...")
            time.sleep(backoff)
    trajectory.append("gave_up")
    return None, trajectory


def extractive_answer_with_retry(question: str, retrieved_chunks: List[dict], flake_chance: float = 0.0):
    """Wraps extractive_lookup_answer so it can be retried like any other
    pipeline step. flake_chance simulates an occasional transient failure."""
    def attempt():
        if random.random() < flake_chance:
            raise TransientLocalError("simulated transient failure in the local pipeline")
        answer = extractive_lookup_answer(question, retrieved_chunks)
        return answer or "I couldn't find a sentence that answers that question in the retrieved context."

    return call_with_retry(attempt, max_retries=3)


demo_question = "What is this document about?"
top_chunks = sorted(
    pdf_chunks, key=lambda c: score_chunk(c["text"], demo_question), reverse=True
)[:4]
answer, trajectory = extractive_answer_with_retry(demo_question, top_chunks, flake_chance=0.5)
print("\nAnswer:", answer)
print("Trajectory:", trajectory)



Answer: I couldn't find a sentence that answers that question in the retrieved context.
Trajectory: ['attempt_1', 'success']


## 5. Single Agent Simulating Multi-Agent Behavior

A single agent can act like several specialized agents by splitting its work
into distinct **roles** — e.g. a "Retriever", a "Router", and an "Answerer" —
all executed inside one process. This keeps the system simple while still
getting the benefits of role separation.


In [ ]:
class SinglePdfAgent:
    """One agent, three internal 'roles' that behave like separate sub-agents."""

    def __init__(self, chunks: List[dict]):
        self.chunks = chunks

    def router_role(self, question: str) -> str:
        route = route_question(question)
        print(f"  [Router role] classified as: {route}")
        return route

    def retriever_role(self, question: str, top_k: int = 4) -> List[dict]:
        scored = sorted(self.chunks, key=lambda c: score_chunk(c["text"], question), reverse=True)
        top = [c for c in scored[:top_k] if score_chunk(c["text"], question) > 0]
        print(f"  [Retriever role] retrieved {len(top)} chunk(s)")
        return top

    def answerer_role(self, question: str, route: str, context_chunks: List[dict]) -> str:
        if route == "Out of Scope":
            answer = "That question is outside the scope of this PDF."
        elif route == "Summary":
            answer = extractive_summary(self.chunks)
        elif not context_chunks:
            answer = "I couldn't find anything relevant to that question in the PDF."
        else:
            answer, _ = extractive_answer_with_retry(question, context_chunks)
        print(f"  [Answerer role] produced an answer ({len(answer)} chars)")
        return answer

    def handle(self, question: str) -> str:
        route = self.router_role(question)
        context_chunks = [] if route in ("Out of Scope", "Summary") else self.retriever_role(question)
        return self.answerer_role(question, route, context_chunks)


def extractive_summary(chunks: List[dict], num_sentences: int = 3) -> str:
    """A frequency-based extractive summary: score every sentence by how many
    'important' words it shares with the rest of the document (classic
    frequency-based summarization), then keep the top-scoring sentences in
    their original reading order."""
    all_sentences = []
    seen_sentences = set()  # chunks can overlap, so dedupe repeated sentences
    for c in chunks:
        for sentence in split_into_sentences(c["text"]):
            key = sentence.strip().lower()
            if key in seen_sentences:
                continue
            seen_sentences.add(key)
            all_sentences.append((c["page"], sentence))

    if not all_sentences:
        return "The document doesn't contain enough extractable text to summarize."

    freq = word_frequencies([s for _, s in all_sentences])
    ranked_indices = sorted(
        range(len(all_sentences)),
        key=lambda i: score_sentence(all_sentences[i][1], freq),
        reverse=True,
    )
    top_indices = sorted(ranked_indices[:num_sentences])  # restore original order
    return " ".join(f"[Page {all_sentences[i][0]}] {all_sentences[i][1]}" for i in top_indices)


agent = SinglePdfAgent(pdf_chunks)
print(agent.handle("Can you summarize this document?"))


  [Router role] classified as: Summary
  [Answerer role] produced an answer (177 chars)
[Page 1] What is conditional routing in an agent system? [Page 2] Compare sequential tool calls and parallel tool calls. [Page 2] What is trajectory evaluation in agent systems?


## 6. JSON Schema Tools

Tools are more reliable when their inputs are validated against a **JSON
schema** before use. Here, the agent's "retrieve" tool takes a structured
payload (`question`, `top_k`) and we validate it before running the retrieval,
just like you would validate arguments to any real tool call.


In [ ]:
RETRIEVE_TOOL_SCHEMA = {
    "type": "object",
    "properties": {
        "question": {"type": "string", "minLength": 1},
        "top_k": {"type": "integer", "minimum": 1, "maximum": 10},
    },
    "required": ["question"],
}


def validate_against_schema(payload: dict, schema: dict) -> List[str]:
    """A tiny hand-rolled JSON-schema validator (no external deps needed)."""
    errors = []
    for field_name in schema.get("required", []):
        if field_name not in payload:
            errors.append(f"missing required field: {field_name}")

    for field_name, rules in schema.get("properties", {}).items():
        if field_name not in payload:
            continue
        value = payload[field_name]
        expected_type = rules.get("type")
        if expected_type == "string" and not isinstance(value, str):
            errors.append(f"{field_name} should be a string")
        if expected_type == "integer" and not isinstance(value, int):
            errors.append(f"{field_name} should be an integer")
        if "minLength" in rules and isinstance(value, str) and len(value) < rules["minLength"]:
            errors.append(f"{field_name} shorter than minLength {rules['minLength']}")
        if "minimum" in rules and isinstance(value, int) and value < rules["minimum"]:
            errors.append(f"{field_name} below minimum {rules['minimum']}")
        if "maximum" in rules and isinstance(value, int) and value > rules["maximum"]:
            errors.append(f"{field_name} above maximum {rules['maximum']}")
    return errors


def retrieve_tool(payload: dict, chunks: List[dict]):
    errors = validate_against_schema(payload, RETRIEVE_TOOL_SCHEMA)
    if errors:
        return {"status": "error", "errors": errors}

    top_k = payload.get("top_k", 4)
    scored = sorted(chunks, key=lambda c: score_chunk(c["text"], payload["question"]), reverse=True)
    top = [c for c in scored[:top_k] if score_chunk(c["text"], payload["question"]) > 0]
    return {"status": "ok", "chunks": top}


print(retrieve_tool({"question": "What is this document about?", "top_k": 3}, pdf_chunks))
print(retrieve_tool({"top_k": 3}, pdf_chunks))          # missing required field
print(retrieve_tool({"question": "x", "top_k": 99}, pdf_chunks))  # top_k out of range


{'status': 'ok', 'chunks': []}
{'status': 'error', 'errors': ['missing required field: question']}
{'status': 'error', 'errors': ['top_k above maximum 10']}


## 7. Sequential vs. Parallel Tool Calls

When an agent needs to run several independent calls (e.g. retrieving and
extracting answers for several unrelated questions at once), running them **in
parallel** can be faster than running them **sequentially** — as long as the
calls don't depend on each other's output.

Note: our extractive answer function is pure local computation, so it's
already extremely fast -- to make the difference visible, we add a small
artificial delay per call to stand in for the kind of per-question work (e.g.
scanning a much larger document) where parallelism would actually matter.


In [ ]:
questions_batch = [
    "What is this document about?",
    "Does the document mention any dates?",
    "What is the main conclusion or takeaway?",
]

ARTIFICIAL_DELAY_SEC = 0.3  # stands in for heavier per-question work

def answer_one(question: str) -> str:
    time.sleep(ARTIFICIAL_DELAY_SEC)
    top = sorted(pdf_chunks, key=lambda c: score_chunk(c["text"], question), reverse=True)[:4]
    top = [c for c in top if score_chunk(c["text"], question) > 0]
    if not top:
        return "No relevant context found."
    answer, _ = extractive_answer_with_retry(question, top)
    return answer

# Sequential
start = time.time()
sequential_answers = [answer_one(q) for q in questions_batch]
sequential_time = time.time() - start

# Parallel
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=len(questions_batch)) as executor:
    parallel_answers = list(executor.map(answer_one, questions_batch))
parallel_time = time.time() - start

print(f"Sequential time: {sequential_time:.2f}s")
print(f"Parallel time:   {parallel_time:.2f}s\n")
for q, a in zip(questions_batch, parallel_answers):
    print(f"Q: {q}\nA: {a}\n")

print("-> Prefer sequential when questions depend on each other; parallel when they don't.")


Sequential time: 0.90s
Parallel time:   0.30s

Q: What is this document about?
A: No relevant context found.

Q: Does the document mention any dates?
A: No relevant context found.

Q: What is the main conclusion or takeaway?
A: No relevant context found.

-> Prefer sequential when questions depend on each other; parallel when they don't.


## 8. Error Handling Strategies

Two common strategies for a tool-using agent:

1. **try/except with meaningful error messages** — catch failures gracefully
   instead of crashing (e.g. a corrupt/unreadable PDF, or an empty PDF with no
   extractable text).
2. **automatic retries** — re-attempt transient failures (as in Section 4).

Logging is added here as a lightweight third layer, useful for
debugging/monitoring.


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("pdf_agent")


def safe_pdf_load(path: str):
    """Strategy 1: try/except with a meaningful, structured error message."""
    try:
        reader = PdfReader(path)
        pages = [p.extract_text() or "" for p in reader.pages]
        if not any(p.strip() for p in pages):
            raise ValueError("PDF contains no extractable text (it may be a scanned image).")
        return {"status": "ok", "pages": pages}
    except Exception as e:
        logger.error(f"PDF load failed: {e}")
        return {"status": "error", "message": str(e)}


print(safe_pdf_load(PDF_PATH))
print(safe_pdf_load("does_not_exist.pdf"))   # triggers a clear, handled error

# Strategy 2: retry mechanism (reusing call_with_retry from Section 4) applied
# to our extractive-answer step, simulated as occasionally flaky.
random.seed(3)
result, trajectory = call_with_retry(
    lambda: extractive_answer_with_retry("What is this document about?", pdf_chunks[:2], flake_chance=0.6)[0],
    max_retries=4,
)
print("\nRetry trajectory:", trajectory)


ERROR:pdf_agent:PDF load failed: [Errno 2] No such file or directory: 'does_not_exist.pdf'


{'status': 'ok', 'pages': ['Single  Agent  Systems  &  Agent  Pipelines  \n-\n \nQuiz\n \n \nInstructions:   \nAnswer\n \nthe\n \nfollowing\n \nquestions.\n \nThese\n \nquestions\n \nare\n \ndesigned\n \nto\n \ntest\n \nyour\n \nunderstanding\n \nof\n \nagent\n \nsystems,\n \ntools,\n \nand\n \nevaluation.\n \n \n \n \n1.  Explain  the  concept  of  a  stateful  directed  graph  in  agent  pipelines.  How  does  it  differ  from  a  simple  \nlinear\n \npipeline?\n \n \nA  stateful  directed  graph  is  a  workflow  where  each  step  can  store  and  use  information  from  previous  \nsteps.\n \nThe\n \nworkflow\n \nis\n \nmade\n \nup\n \nof\n \nnodes\n \nand\n \nedges\n \nthat\n \ndefine\n \nhow\n \ndata\n \nmoves\n \nthrough\n \nthe\n \nsystem.\n \nIt\n \ncan\n \nsupport\n \nbranching,\n \nlooping,\n \nand\n \ndecision-making.\n \nThis\n \nmakes\n \nit\n \nmore\n \nflexible\n \nand\n \nintelligent.\n \nIn\n \ncontrast,\n \na\n \nlinear\n \npipeline\n \nfollows\n \na\n \nfixed\n \ns

## 9. Trajectory Evaluation

**Trajectory evaluation** looks at the *entire path* an agent took — every node
visited — not just whether the final answer looked reasonable. This reveals
inefficient or risky behavior (e.g. skipping retrieval, extracting an answer
with no context, looping too many times) that a final-answer-only check would
miss.


In [ ]:
def evaluate_trajectory(trajectory: List[str], expected_final_node: str) -> dict:
    used_expected_node = expected_final_node in trajectory
    num_steps = len(trajectory)
    took_reasonable_path = num_steps <= 2  # retrieve + answer, e.g.

    return {
        "used_expected_node": used_expected_node,
        "num_steps": num_steps,
        "efficient": took_reasonable_path,
        "trajectory": trajectory,
    }


# Re-run our node/edge graph from Section 2 and evaluate its trajectory
out = node_edge_graph.run({"question": "What is this document about?", "chunks": pdf_chunks})
evaluation = evaluate_trajectory(out["trajectory"], expected_final_node="extractive_answer")
print(json.dumps(evaluation, indent=2))


  [node] retrieve_context -> found 0 relevant chunk(s)
  [node] no_context -> nothing relevant found
{
  "used_expected_node": false,
  "num_steps": 2,
  "efficient": true,
  "trajectory": [
    "retrieve_context",
    "no_context"
  ]
}


## 10. Task Completion Rate & Compute Metrics

- **Task completion rate** = percentage of questions the agent successfully
  answers (i.e. doesn't give up after retries).
- **Compute metrics** = resources spent per question -- since there's no paid
  API here, "cost" becomes number of local answer-function calls and time
  taken (still useful for spotting an inefficient or overly retry-happy
  pipeline).

The simulation below asks the agent several questions and reports both
metrics.


In [ ]:
def run_pdf_qa_batch(questions: List[str], chunks: List[dict], max_retries: int = 3):
    completed = 0
    total_calls = 0
    total_time = 0.0

    for q in questions:
        start = time.time()
        top = sorted(chunks, key=lambda c: score_chunk(c["text"], q), reverse=True)[:4]
        top = [c for c in top if score_chunk(c["text"], q) > 0]

        if not top:
            elapsed = time.time() - start
            total_time += elapsed
            continue  # no context -> not counted as a completed Q&A task

        answer, trajectory = extractive_answer_with_retry(q, top, flake_chance=0.2)
        elapsed = time.time() - start
        total_time += elapsed
        total_calls += len([t for t in trajectory if t.startswith("attempt_")])
        if answer is not None:
            completed += 1

    n = len(questions)
    return {
        "task_completion_rate": completed / n,
        "avg_calls_per_task": total_calls / n,
        "avg_time_per_task_sec": round(total_time / n, 4),
    }


sample_questions = [
    "What is this document about?",
    "What is the main conclusion?",
    "Are there any numbers or statistics mentioned?",
]

metrics = run_pdf_qa_batch(sample_questions, pdf_chunks)
print(json.dumps(metrics, indent=2))

print("""
Optimization ideas suggested by these numbers:
- If completion rate is low     -> improve retrieval (bigger top_k, better chunking).
- If avg_calls_per_task is high -> the flaky-step simulation is forcing retries; investigate.
- If avg_time_per_task is high  -> consider parallelizing independent questions (see Section 7).
""")


{
  "task_completion_rate": 0.0,
  "avg_calls_per_task": 0.0,
  "avg_time_per_task_sec": 0.0007
}

Optimization ideas suggested by these numbers:
- If completion rate is low     -> improve retrieval (bigger top_k, better chunking).
- If avg_calls_per_task is high -> the flaky-step simulation is forcing retries; investigate.
- If avg_time_per_task is high  -> consider parallelizing independent questions (see Section 7).



## Putting It All Together: The PDF Q&A Agent Pipeline

A complete pipeline that combines: a stateful graph (1), nodes/edges (2),
conditional routing (3), a retry loop (4), JSON-schema-validated tool calls (6),
error handling (8), and trajectory evaluation (9) — all wired into one agent
that answers questions grounded in your PDF, using **only local, from-scratch
extractive Q&A -- no API key, no network calls.**

Run the setup cells above once (PDF load included), then run this cell and ask
as many questions as you like.


In [ ]:
def build_pdf_qa_pipeline(chunks: List[dict]) -> StateGraph:

    def classify(state):
        state["route"] = route_question(state["question"])
        return state

    def retrieve(state):
        payload = {"question": state["question"], "top_k": 4}
        tool_result = retrieve_tool(payload, chunks)
        if tool_result["status"] == "error":
            state["retrieved"] = []
            state["tool_errors"] = tool_result["errors"]
        else:
            state["retrieved"] = tool_result["chunks"]
        return state

    def answer_summary(state):
        state["answer"] = extractive_summary(chunks)
        return state

    def answer_lookup(state):
        def attempt():
            answer = extractive_lookup_answer(state["question"], state["retrieved"])
            return answer or "I couldn't find a sentence that answers that question in the retrieved context."

        result, retry_trace = call_with_retry(attempt, max_retries=3)
        state["answer"] = result if result is not None else (
            "Sorry, I couldn't extract an answer after repeated attempts -- please try again."
        )
        state["retry_trace"] = retry_trace
        return state

    def answer_out_of_scope(state):
        state["answer"] = "That question is outside the scope of this PDF."
        return state

    def answer_no_context(state):
        state["answer"] = "I couldn't find anything relevant to that question in the PDF."
        return state

    def router(state):
        if state["route"] == "Out of Scope":
            return "out_of_scope"
        if state["route"] == "Summary":
            return "answer_summary"
        return "retrieve"

    def router_after_retrieve(state):
        return "answer_lookup" if state["retrieved"] else "no_context"

    g = StateGraph()
    g.add_node("classify", classify)
    g.add_node("retrieve", retrieve)
    g.add_node("answer_summary", answer_summary)
    g.add_node("answer_lookup", answer_lookup)
    g.add_node("out_of_scope", answer_out_of_scope)
    g.add_node("no_context", answer_no_context)
    g.add_edge("classify", router)
    g.add_edge("retrieve", router_after_retrieve)
    g.add_edge("answer_summary", lambda s: "END")
    g.add_edge("answer_lookup", lambda s: "END")
    g.add_edge("out_of_scope", lambda s: "END")
    g.add_edge("no_context", lambda s: "END")
    g.set_entry("classify")
    return g


pdf_qa_pipeline = build_pdf_qa_pipeline(pdf_chunks)

demo_questions = [
    "Can you summarize this document?",
    "What is the main topic discussed?",
    "What's your favorite color?",
]

for question in demo_questions:
    print(f"\n=== Question: {question!r} ===")
    out = pdf_qa_pipeline.run({"question": question})
    print("Answer:", out["answer"])
    print("Trajectory:", out["trajectory"])
    print("Evaluation:", evaluate_trajectory(out["trajectory"], expected_final_node=out["trajectory"][-1]))



=== Question: 'Can you summarize this document?' ===
Answer: [Page 1] What is conditional routing in an agent system? [Page 2] Compare sequential tool calls and parallel tool calls. [Page 2] What is trajectory evaluation in agent systems?
Trajectory: ['classify', 'answer_summary']
Evaluation: {'used_expected_node': True, 'num_steps': 2, 'efficient': True, 'trajectory': ['classify', 'answer_summary']}

=== Question: 'What is the main topic discussed?' ===
Answer: I couldn't find anything relevant to that question in the PDF.
Trajectory: ['classify', 'retrieve', 'no_context']
Evaluation: {'used_expected_node': True, 'num_steps': 3, 'efficient': False, 'trajectory': ['classify', 'retrieve', 'no_context']}

=== Question: "What's your favorite color?" ===
Answer: I couldn't find anything relevant to that question in the PDF.
Trajectory: ['classify', 'retrieve', 'no_context']
Evaluation: {'used_expected_node': True, 'num_steps': 3, 'efficient': False, 'trajectory': ['classify', 'retrieve', 

### Ask your own question

Run this cell as many times as you like -- it reuses the PDF already loaded and
chunked above, and never makes a network call.


In [ ]:
your_question = input("Ask a question about the PDF: ")
out = pdf_qa_pipeline.run({"question": your_question})
print("\nAnswer:", out["answer"])
print("Trajectory:", out["trajectory"])


Ask a question about the PDF: Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?

Answer: [Page 1] Explain the concept of a stateful directed graph in agent pipelines. [Page 1] How does it differ from a simple linear pipeline?
Trajectory: ['classify', 'retrieve', 'answer_lookup']
